# LangChain Agents — Detailed Client Reference Notebook

This notebook explains the LangChain `create_agent()` framework in a practical, step-by-step manner. It is designed as a learning/reference document for clients, architects, and developers.

## 1. What is an Agent?

An agent is:

**Agent = Model + Harness**

The model generates reasoning and decisions.
The harness provides prompts, tools, memory, middleware, execution flow, and guardrails.

Typical loop:
1. Receive user request
2. Decide whether to use tools
3. Execute tools
4. Observe results
5. Continue until task completion
6. Return final answer

In [ ]:
# Basic Agent Creation

from langchain.agents import create_agent

agent = create_agent(
    model='openai:gpt-5.4',
    tools=[]
)

# The harness is automatically created around the model.

## 2. Core Components

### Model
Responsible for reasoning and generation.

### Tools
Allow external actions such as search, APIs, databases, calculators.

### System Prompt
Defines behavior, personality, and constraints.

### Structured Output
Returns validated schemas instead of free-form text.

### Name
Useful when embedding agents inside multi-agent systems.

In [ ]:
from langchain.tools import tool

@tool
def search(query:str)->str:
    '''Search for information.'''
    return f'Results for: {query}'

agent = create_agent(
    model='openai:gpt-5.4',
    tools=[search],
    system_prompt='Be concise and accurate.'
)

## 3. Structured Output

Use Pydantic models when applications require predictable responses.

In [ ]:
from pydantic import BaseModel

class Answer(BaseModel):
    summary: str
    confidence: float

agent = create_agent(
    'openai:gpt-5.4',
    tools=[],
    response_format=Answer
)

## 4. Conversation Memory with thread_id

LangChain separates:

- Conversation State → thread_id
- Runtime Data → context

thread_id enables checkpointing and conversation continuation.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.utils.uuid import uuid7

agent = create_agent(
    model='openai:gpt-5.4',
    tools=[],
    checkpointer=InMemorySaver()
)

config = {
    'configurable': {
        'thread_id': str(uuid7())
    }
}

## 5. Runtime Context

Runtime context carries user-specific data for a single execution.

Examples:
- User IDs
- Feature flags
- Tenant information
- API keys

This is different from conversation memory.

In [ ]:
from dataclasses import dataclass

@dataclass
class Context:
    user_id: str

# Passed at invoke time
context = Context(user_id='user-123')

## 6. Streaming

Streaming allows users to observe intermediate reasoning steps, tool calls, and progress.

Benefits:
- Better UX
- Faster perceived response times
- Debugging visibility

In [ ]:
for chunk in agent.stream(
    {
        'messages':[{
            'role':'user',
            'content':'Research AI trends'
        }]
    },
    stream_mode='values'
):
    print(chunk)

# 7. Middleware Architecture

Middleware is the primary customization mechanism.

It intercepts different phases of the agent lifecycle.

## Execution Environment Middleware

Provides:
- Filesystem access
- Code execution
- Sandboxes
- Persistent workspace

Useful for coding agents and research agents.

In [ ]:
from deepagents.middleware import FilesystemMiddleware

agent = create_agent(
    model='anthropic:claude-sonnet-4-6',
    tools=[search],
    middleware=[FilesystemMiddleware()]
)

## Context Management Middleware

### SummarizationMiddleware
Compresses history.

### MemoryMiddleware
Loads persistent knowledge.

### SkillsMiddleware
Loads reusable capabilities on demand.

## Planning & Delegation

Large tasks can be split among specialized subagents.

Benefits:
- Parallel work
- Cleaner context windows
- Better scalability

In [ ]:
researcher = {
    'name':'researcher',
    'description':'Search and summarize',
    'tools':[search]
}

## Fault Tolerance

Production systems face:
- Timeouts
- Rate limits
- Temporary API failures

Retry middleware handles these automatically.

In [ ]:
from langchain.agents.middleware import (
    ModelRetryMiddleware,
    ToolRetryMiddleware
)

middleware=[
    ModelRetryMiddleware(max_retries=3),
    ToolRetryMiddleware(max_retries=2)
]

## Guardrails

Guardrails enforce deterministic policies.

Example:
- PII detection
- Compliance checks
- Content filtering

In [ ]:
from langchain.agents.middleware import PIIMiddleware

agent = create_agent(
    model='anthropic:claude-sonnet-4-6',
    tools=[search],
    middleware=[PIIMiddleware()]
)

## Human-in-the-Loop

Allows approval before critical actions.

Common use cases:
- File deletion
- Database updates
- Financial actions
- Customer communications

In [ ]:
from langchain.agents.middleware import HumanInTheLoopMiddleware

agent = create_agent(
    model='anthropic:claude-sonnet-4-6',
    tools=[search],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={'write_file': True}
        )
    ]
)

# 8. Deep Agents

`create_deep_agent()` is a higher-level framework that combines:

- Filesystem
- Summarization
- Subagents
- Prompt caching
- Long-running execution support

Recommended for research and coding workflows.

# 9. Architectural Summary

Layer 1: Model

Layer 2: Harness (`create_agent`)

Layer 3: Tools

Layer 4: Middleware

Layer 5: Memory & Persistence

Layer 6: Human Oversight

This layered architecture is what makes LangChain agents production-ready.

# 10. Client Takeaways

1. `create_agent()` is the central abstraction.
2. Middleware provides enterprise-grade extensibility.
3. thread_id manages conversation continuity.
4. context handles runtime metadata.
5. Tools provide external capabilities.
6. Structured output improves reliability.
7. Subagents enable scale.
8. Guardrails and HITL improve safety.
9. Deep Agents are suitable for complex autonomous workflows.
10. LangGraph powers the execution and persistence layer underneath.